# Capítulo 12: Regressão Múltipla

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto. É o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem daqui —
# no livro isso vem do `execute-dir: project` do Quarto.
import os
import sys

_raiz = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_raiz, "_quarto.yml")):
    _pai = os.path.dirname(_raiz)
    if _pai == _raiz:
        raise RuntimeError("raiz do projeto não encontrada (procurando _quarto.yml)")
    _raiz = _pai
os.chdir(_raiz)
if _raiz not in sys.path:
    sys.path.insert(0, _raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 15 de Grus (2019).

> Eu não olho para um problema e coloco nele variáveis que não o afetam.
>
> — Bill Parcells

O [Capítulo 11](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/index.html) previu quantos minutos por dia um usuário passa no site a partir de uma única informação: quantos amigos ele tem. Funcionou — a reta capturou uma tendência real —, mas capturou só um terço da variação. Os outros dois terços vieram de coisas que aquele modelo não tinha como enxergar, simplesmente porque não estavam nele.

Agora estão. Sabemos também quantas horas por dia cada usuário trabalha e se ele tem doutorado. Este capítulo estende o modelo de uma variável explicativa para várias, e a extensão é quase indolor: o `predict` vira um produto escalar, o gradiente vira uma lista, e o gradiente descendente do [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html) continua funcionando sem uma linha de mudança. O R² sobe de 0,33 para 0,68.

E é aí que o capítulo fica interessante, porque essa subida esconde uma armadilha. **O R² nunca desce quando você acrescenta uma variável** — nem quando a variável é uma coluna de números aleatórios sem relação nenhuma com o alvo. A métrica que usamos para julgar o modelo é incapaz, por construção, de dizer que uma variável não deveria estar ali. Metade deste capítulo é a resposta a esse problema: uma técnica de reamostragem que mede incerteza sem exigir teoria estatística nenhuma (o **bootstrap**), o que ela revela sobre cada coeficiente (os **erros padrão**) e o que fazer com um coeficiente que não se sustenta (a **regularização**). As quatro últimas seções são uma única linha de raciocínio, não quatro tópicos.

Ao final deste capítulo, você será capaz de:

- Escrever o modelo de regressão múltipla como um produto escalar e explicar por que a primeira coluna é sempre 1
- Nomear as hipóteses que o modelo de mínimos quadrados exige e dizer **o que quebra** quando cada uma é violada
- Ajustar o modelo por gradiente descendente, reaproveitando sem alteração o otimizador do Capítulo 5
- Interpretar um coeficiente como um efeito "mantendo todo o resto constante" — e reconhecer quando essa frase descreve uma situação que não existe nos dados
- Demonstrar, rodando, que o R² sobe ao acrescentar variáveis puramente aleatórias
- Estimar a incerteza de qualquer estatística por bootstrap, sem fórmula fechada
- Usar erros padrão e valores-p para separar coeficientes reais de coeficientes que são ruído
- Aplicar regularização *ridge* e explicar por que ela ataca exatamente o problema que o R² esconde

## Seções

| Seção | Tópico |
|---|---|
| [12.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/01-o-modelo.html) | O Modelo |
| [12.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/02-hipoteses-do-minimos-quadrados.html) | Outras Hipóteses do Modelo de Mínimos Quadrados |
| [12.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/03-ajustando-o-modelo.html) | Ajustando o Modelo |
| [12.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/04-interpretando-o-modelo.html) | Interpretando o Modelo |
| [12.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/05-qualidade-do-ajuste.html) | Qualidade do Ajuste |
| [12.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/06-digressao-o-bootstrap.html) | Digressão: O Bootstrap |
| [12.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/07-erros-padrao-dos-coeficientes.html) | Erros Padrão dos Coeficientes |
| [12.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/08-regularizacao.html) | Regularização |

## O Modelo

> **📌 Nota**
>
> Esta seção corresponde a *The Model*, do capítulo 15 de Grus (2019).

A vice-presidente da DataSciencester ficou impressionada com o modelo do [Capítulo 11](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/index.html), mas acha que dá para fazer melhor. E coletou mais dados para provar: agora sabemos, para cada usuário, **quantas horas por dia ele trabalha** e **se ele tem doutorado**.

O modelo do capítulo anterior tinha uma variável explicativa:

$$
y_i = \alpha + \beta x_i + \varepsilon_i
$$

A generalização é imediata. Em vez de um único número $x_i$, cada entrada passa a ser um vetor de $k$ números, $x_{i1}, \ldots, x_{ik}$, e cada um deles ganha o próprio coeficiente:

$$
y_i = \alpha + \beta_1 x_{i1} + \cdots + \beta_k x_{ik} + \varepsilon_i
$$

No nosso caso concreto:

$$
\text{minutos} = \alpha + \beta_1 \,\text{amigos} + \beta_2 \,\text{horas de trabalho} + \beta_3 \,\text{doutorado} + \varepsilon
$$

### Uma variável que não é um número

"Tem doutorado" não é uma quantidade — é sim ou não. Isso não é obstáculo: representamos a resposta como uma **variável indicadora** (o termo em inglês, *dummy variable*, é o que você vai encontrar na literatura), que vale 1 para quem tem doutorado e 0 para quem não tem. A partir daí ela é tão numérica quanto as outras, e o modelo não sabe nem precisa saber que aquela coluna representa uma categoria.

> **📌 Nota**
>
> Duas armadilhas que essa codificação abre, e que vão importar mais adiante neste capítulo:
>
> - O coeficiente $\beta_3$ passa a ter uma leitura bem específica: é o efeito de **mudar de 0 para 1**, ou seja, a diferença média entre quem tem e quem não tem doutorado. Não faz sentido falar em "meio doutorado".
> - Se a categoria tivesse **três** valores em vez de dois — digamos, graduação, mestrado e doutorado —, uma única coluna com os códigos 0, 1 e 2 seria errada: ela imporia que a diferença entre graduação e mestrado é exatamente igual à diferença entre mestrado e doutorado, o que não há razão para supor. O caminho correto é uma coluna indicadora para cada categoria, **menos uma**. O porquê desse "menos uma" é o assunto da [seção 12.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/02-hipoteses-do-minimos-quadrados.html).

### O truque da coluna de 1

Em regressão múltipla, o conjunto de parâmetros é tratado como um vetor único, chamado $\beta$. Queremos que ele inclua também o termo constante — o que no capítulo anterior se chamava $\alpha$ —, e o jeito de conseguir isso é acrescentar aos dados uma coluna cheia de 1:

```python
beta = [alpha, beta_1, ..., beta_k]
x_i  = [1, x_i1, ..., x_ik]
```

Com essa convenção, $\alpha \cdot 1$ é só mais um termo da soma, e o modelo inteiro vira um **produto escalar** — a função `dot` que o [Capítulo 4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/index.html) construiu, e que continua sendo `sum(v_i * w_i for v_i, w_i in zip(v, w))`, sem numpy nenhum:

In [ ]:
from scratch.linear_algebra import dot, Vector

def predict(x: Vector, beta: Vector) -> float:
    """assume que o primeiro elemento de x é 1"""
    return dot(x, beta)

Duas linhas. O modelo de regressão múltipla inteiro é um produto escalar entre a linha de dados e o vetor de parâmetros — e todo o resto deste capítulo é sobre como escolher esse vetor e o quanto confiar nele depois de escolhido.

> **🔷 Conceito**
>
> A coluna de 1 não é um detalhe de implementação: é o que torna o intercepto **um coeficiente como qualquer outro**. Sem ela, `predict` teria que somar `alpha` separadamente, o gradiente teria um caso especial, e cada função escrita a partir daqui carregaria essa exceção. Com ela, existe só uma regra — produto escalar — e ela vale para o vetor inteiro.
>
> O preço é lembrar da convenção. Passar a `predict` uma linha de dados sem o 1 na frente não gera erro nenhum: gera uma previsão silenciosamente errada, com o intercepto multiplicando a primeira variável de verdade.

### Os dados

Cada entrada, então, é uma lista de quatro números:

```python
[1,    # termo constante
 49,   # número de amigos
 4,    # horas de trabalho por dia
 0]    # não tem doutorado
```

O pacote `scratch` já traz as 203 linhas prontas, no mesmo formato:

In [ ]:
import random
random.seed(0)
from scratch.multiple_regression import inputs
from scratch.statistics import daily_minutes_good
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
len(inputs), len(inputs[0]), inputs[0]

In [ ]:
len(daily_minutes_good), daily_minutes_good[0]

São os **mesmos 203 usuários** do Capítulo 11 — `daily_minutes_good` é exatamente a mesma lista de minutos por dia que aquele capítulo tentou prever, e a segunda coluna de `inputs` é exatamente `num_friends_good`. O que mudou não foi o dado a ser previsto; foi a quantidade de informação disponível para prevê-lo.

> **💡 Dica — Na prática: `scikit-learn`**
>
> A coluna de 1 é uma convenção do livro, não uma exigência da matemática, e a maior parte das bibliotecas prefere escondê-la:
>
> ```python
> from sklearn.linear_model import LinearRegression
>
> X = [x[1:] for x in inputs]     # SEM a coluna de 1
> y = daily_minutes_good
>
> modelo = LinearRegression().fit(X, y)   # fit_intercept=True é o padrão
> modelo.intercept_, modelo.coef_
> ```
>
> `LinearRegression` mantém `fit_intercept=True` por padrão e cuida do termo constante internamente, o que separa `intercept_` (um escalar) de `coef_` (o vetor dos demais). Passar a ela uma matriz que **já** tem a coluna de 1 é um erro comum e silencioso: o modelo acaba com duas colunas constantes, e a [seção 12.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/02-hipoteses-do-minimos-quadrados.html) explica exatamente o que isso quebra.
>
> Para a variável indicadora, o equivalente industrial é `pandas.get_dummies` ou o `OneHotEncoder` do `scikit-learn`, que recebem uma coluna de categorias e devolvem as colunas de 0 e 1. Ambos aceitam `drop_first=True` / `drop='first'` — o "menos uma" mencionado acima, que não é opcional quando o modelo tem intercepto.

## Outras Hipóteses do Modelo de Mínimos Quadrados

> **📌 Nota**
>
> Esta seção corresponde a *Further Assumptions of the Least Squares Model*, do capítulo 15 de Grus (2019).

Passar de uma variável explicativa para várias exige duas hipóteses adicionais para que o modelo — e a nossa solução para ele — façam sentido. Listá-las é fácil e inútil; o que interessa é **o que quebra quando cada uma é violada**, e as duas quebram de maneiras bem diferentes. A primeira torna a resposta indeterminada. A segunda dá uma resposta errada com toda a aparência de estar certa.

In [ ]:
import random
random.seed(0)
from scratch.multiple_regression import inputs, least_squares_fit, multiple_r_squared
from scratch.statistics import daily_minutes_good, correlation
import matplotlib.pyplot as plt
plt.close('all')

### Hipótese 1: as colunas de `x` são linearmente independentes

Nenhuma coluna pode ser escrita como soma ponderada das outras. Quando essa hipótese falha, **é impossível estimar `beta`** — e o motivo não é numérico, é lógico: não existe *uma* resposta a ser encontrada.

O caso extremo torna isso visível. Imagine que os dados tivessem um campo extra, `num_conhecidos`, que por um acidente de coleta fosse **exatamente igual** a `num_amigos` para todo usuário. Vamos construir esse campo e ajustar o modelo três vezes, cada uma partindo de um chute inicial diferente:

In [ ]:
inputs_colineares = [x + [x[1]] for x in inputs]   # num_conhecidos == num_amigos

for semente in [0, 1, 2]:
    random.seed(semente)
    b = least_squares_fit(inputs_colineares, daily_minutes_good, 0.001, 5000, 25)
    r2 = multiple_r_squared(inputs_colineares, daily_minutes_good, b)
    print(f"semente {semente}: amigos = {b[1]:.4f}   conhecidos = {b[4]:.4f}   "
          f"soma = {b[1] + b[4]:.4f}   R² = {r2:.6f}")

Três ajustes, três respostas **diferentes** para o coeficiente de `num_amigos` — e todas igualmente boas. Repare no que é idêntico nas três linhas: a **soma** dos dois coeficientes e o R². O modelo determina perfeitamente quanto vale `amigos + conhecidos`; ele não tem como decidir onde termina a parte de um e começa a do outro, porque nenhum dado distingue as duas hipóteses.

> **🔷 Conceito**
>
> Partindo de qualquer `beta`, some um valor qualquer ao coeficiente de `num_amigos` e subtraia o mesmo valor do coeficiente de `num_conhecidos`. Como as duas colunas são idênticas, as previsões do modelo não mudam **nem um pouco** — e portanto o erro não muda. Existe uma infinidade de vetores `beta` empatados no primeiro lugar, e "o" coeficiente de `num_amigos` simplesmente não existe.
>
> Note que o modelo continua **prevendo** bem: o R² é o mesmo nos três ajustes. O que se perde é a capacidade de **interpretar** os coeficientes, que é justamente o que a [seção 12.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/04-interpretando-o-modelo.html) vai querer fazer. Um modelo pode ser útil para prever e inútil para explicar.

Esse é também o motivo do "menos uma" que a [seção 12.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/01-o-modelo.html) mencionou ao codificar categorias. Se uma variável de três categorias virasse três colunas indicadoras, elas somariam 1 em toda linha — ou seja, reproduziriam exatamente a coluna constante do intercepto. Uma coluna seria combinação linear das outras, e cairíamos nesse mesmo poço. O nome do erro é *dummy variable trap*, e ele é frequente o bastante para ter nome próprio.

Violações **exatas** como essa são raras na prática (e, quando acontecem, quase sempre por descuido: uma coluna duplicada, um total que é a soma das partes, um conjunto completo de indicadoras). O que é comum é a versão aproximada — duas colunas quase redundantes, a chamada **multicolinearidade**. Aí o ajuste é tecnicamente único, mas quase indeterminado: o coeficiente fica extremamente sensível a pequenas mudanças nos dados. Guarde essa frase; a [seção 12.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/07-erros-padrao-dos-coeficientes.html) vai medir exatamente essa sensibilidade, e nossos próprios dados têm um caso:

In [ ]:
amigos = [x[1] for x in inputs]
horas  = [x[2] for x in inputs]
phd    = [x[3] for x in inputs]

print(f"corr(amigos, horas) = {correlation(amigos, horas):.4f}")
print(f"corr(amigos, phd)   = {correlation(amigos, phd):.4f}")
print(f"corr(horas,  phd)   = {correlation(horas, phd):.4f}")

`num_amigos` e `tem_doutorado` têm correlação de $-0{,}61$. Olhando mais de perto, a redundância é ainda maior do que esse número sugere:

In [ ]:
faixa_phd    = [a for a, p in zip(amigos, phd) if p == 1]
faixa_sem    = [a for a, p in zip(amigos, phd) if p == 0]

print(f"com doutorado ({len(faixa_phd):3d} usuários): amigos de {min(faixa_phd)} a {max(faixa_phd)}")
print(f"sem doutorado ({len(faixa_sem):3d} usuários): amigos de {min(faixa_sem)} a {max(faixa_sem)}")
print(f"única sobreposição: usuários com exatamente 6 amigos "
      f"({sum(1 for a, p in zip(amigos, phd) if a == 6 and p == 1)} com doutorado, "
      f"{sum(1 for a, p in zip(amigos, phd) if a == 6 and p == 0)} sem)")

Neste conjunto de dados, saber o número de amigos de um usuário quase determina se ele tem doutorado: abaixo de 6 amigos, tem; acima de 6, não tem. A **única** faixa em que os dois grupos coexistem é a de exatamente 6 amigos — 22 usuários dos 203. `tem_doutorado` não é uma coluna redundante, mas falta pouco. As seções 12.7 e 12.8 vão colher a consequência disso.

### Hipótese 2: as colunas de `x` não são correlacionadas com os erros

Se essa hipótese falha, as estimativas de `beta` ficam **sistematicamente erradas** — e essa é a palavra que assusta. Não é ruído, que some ao coletar mais dados; é viés, que permanece.

O mecanismo é este: quando uma variável que afeta `y` fica **de fora** do modelo, o efeito dela não desaparece — ele vai parar no termo de erro. Se alguma variável que **está** no modelo for correlacionada com a que ficou de fora, o coeficiente dela absorve parte de um efeito que não é seu.

> **🟩 Exemplo**
>
> O exemplo de Grus (2019) é hipotético, mas vale seguir o raciocínio. Suponha que:
>
> - Pessoas que trabalham mais horas passam menos tempo no site.
> - Pessoas com mais amigos tendem a trabalhar mais horas.
>
> Isto é, o modelo "real" incluiria `horas de trabalho` com coeficiente **negativo**, e `horas` seria positivamente correlacionada com `amigos`. Ajuste então o modelo só com `amigos`. As previsões desse modelo, usando o coeficiente "real" de `amigos`, ficariam **altas demais** para quem trabalha muito — porque o desconto das horas de trabalho foi esquecido. E como quem tem muitos amigos trabalha muito, as previsões ficariam altas demais justamente para quem tem muitos amigos.
>
> O ajuste de mínimos quadrados corrige isso do único jeito que pode: **diminuindo** o coeficiente de `amigos`. O resultado é uma estimativa enviesada para baixo — não por erro de conta, mas porque o modelo está fazendo o melhor possível com uma variável faltando.

Nos nossos dados, a correlação entre amigos e horas de trabalho é praticamente nula ($0{,}025$), então essa história específica não se realiza. Mas o **fenômeno** se realiza, por outro caminho — via `tem_doutorado`, que é fortemente correlacionado com `num_amigos`. Dá para ver o coeficiente de `amigos` se mexer conforme mudamos o que mais está no modelo:

In [ ]:
variantes = {
    "amigos":                   [[1.0, x[1]] for x in inputs],
    "amigos + horas":           [[1.0, x[1], x[2]] for x in inputs],
    "amigos + phd":             [[1.0, x[1], x[3]] for x in inputs],
    "amigos + horas + phd":     inputs,
}

for nome, xs in variantes.items():
    random.seed(0)
    b = least_squares_fit(xs, daily_minutes_good, 0.001, 5000, 25)
    print(f"{nome:24s} coeficiente de amigos = {b[1]:.4f}")

O coeficiente de `num_amigos` se move conforme o resto do modelo muda, que é o fenômeno desta seção. Mas os números acima saíram do **nosso** gradiente descendente, e para o modelo de uma variável só ele erra o alvo de forma visível — o callout logo abaixo mostra por quê. Como o argumento aqui é estatístico e não de otimização, vale olhá-lo nos coeficientes **exatos**, os que se obtêm resolvendo a equação normal — o cálculo que a [seção 12.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/03-ajustando-o-modelo.html) discute e que este livro não constrói:

| modelo | coeficiente exato de `amigos` |
|---|---|
| `amigos` | 0,903866 |
| `amigos` + `horas` | 0,927439 |
| `amigos` + `doutorado` | 0,963918 |
| `amigos` + `horas` + `doutorado` | 0,972505 |

Duas coisas nessa tabela. A primeira linha é **exatamente** o $\beta \approx 0{,}9039$ que o [Capítulo 11](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/01-o-modelo.html) obteve pela fórmula fechada — é o mesmo modelo, resolvido de forma exata as duas vezes, então tinha mesmo que bater. E o coeficiente sobe **7,6%** da primeira linha para a última — subindo por qualquer caminho que se tome: acrescentando `horas`, acrescentando `doutorado`, ou os dois. Não é que um deles esteja "errado" e o outro "certo": cada um responde a uma pergunta diferente, e a [seção 12.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/04-interpretando-o-modelo.html) é sobre qual pergunta é essa.

> **⚠️ Atenção — Por que este 0,84 não bate com o 0,904 do Capítulo 11**
>
> O primeiro modelo — só `amigos` — é o mesmo do [Capítulo 11](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/01-o-modelo.html), e a solução exata dele é 0,903866. O nosso gradiente descendente imprimiu 0,8415. A diferença é do **otimizador**, não do modelo.
>
> A tentação é dizer que faltaram passos, ou que a semente foi infeliz. Nenhuma das duas coisas se sustenta, e dá para conferir:

In [ ]:
xs_amigos = [[1.0, x[1]] for x in inputs]

for semente in [0, 1]:
    for passos in [5000, 20000]:
        random.seed(semente)
        b = least_squares_fit(xs_amigos, daily_minutes_good, 0.001, passos, 25)
        print(f"semente {semente}, {passos:5d} passos: {b[1]!r}")

> Quatro ajustes, o mesmo valor **bit a bit** — não "parecido", idêntico até o último dígito da representação. Quadruplicar os passos não move nada, e trocar o chute inicial também não.
>
> O motivo é que o gradiente descendente estocástico com taxa de aprendizado fixa e ordem de lote fixa não pousa no mínimo: ele entra numa **órbita estável deslocada** dele. Como a sequência de lotes se repete igual a cada passagem pelos dados, não há ruído a promediar — mais voltas percorrem a mesma órbita. O viés é estrutural do otimizador com estes hiperparâmetros, e o tamanho dele depende do problema: para o modelo de quatro variáveis, que é o deste capítulo, ele cai para a terceira casa decimal (0,9748 contra 0,9725 exato).
>
> Vale registrar isso em vez de esconder, por dois motivos. Primeiro, porque o **padrão** — o coeficiente de `amigos` subir quando as outras variáveis entram — é robusto e aparece igual na solução exata, que é a da tabela acima. Segundo, porque é um lembrete de que todo número deste capítulo tem duas fontes de incerteza empilhadas: a dos dados e a do otimizador. As seções 12.6 e 12.7 medem a primeira, que é a que interessa. A segunda não é ruído que se promedia: é viés de instrumento, e só some trocando o instrumento — outra taxa de aprendizado, outra ordem de lote, ou a álgebra exata.

> **💡 Dica — Na prática: o que se faz com isso**
>
> Nenhuma biblioteca "resolve" essas duas hipóteses, porque elas não são sobre código — são sobre quais dados você tem. Mas há ferramentas para *detectar* problemas:
>
> Para colinearidade **exata**, o `scikit-learn` não reclama: `LinearRegression` resolve o sistema por decomposição em valores singulares, que devolve a solução de norma mínima em vez de estourar. Você recebe um `beta` bonito, sem aviso nenhum, e ele é apenas um representante arbitrário da infinidade de empatados — exatamente as três respostas diferentes que vimos acima.
>
> Para colinearidade **aproximada**, o diagnóstico usual é o *variance inflation factor* (VIF), disponível no `statsmodels` como `variance_inflation_factor`. Ele mede, para cada coluna, o quanto ela é previsível a partir das outras — que é a versão quantitativa da sobreposição que medimos à mão acima.
>
> Para a segunda hipótese — variáveis correlacionadas com o erro — não existe teste, e não pode existir: o erro é justamente o que você não observou. Detectá-la exige conhecimento do domínio, não estatística. Áreas que dependem disso para valer (econometria, epidemiologia) desenvolveram um arsenal inteiro — variáveis instrumentais, experimentos naturais, desenhos quase-experimentais — e todo esse arsenal existe porque o problema **não** se resolve com mais dados nem com um modelo melhor.

## Ajustando o Modelo

> **📌 Nota**
>
> Esta seção corresponde a *Fitting the Model*, do capítulo 15 de Grus (2019).

Como no modelo linear simples, vamos escolher `beta` para minimizar a soma dos erros ao quadrado. A diferença é que agora não há fórmula fechada que caiba num chunk: o [Capítulo 11](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/01-o-modelo.html) resolveu o caso de uma variável com duas médias, uma correlação e dois desvios padrão, e aquele truque não se estende. Vamos de gradiente descendente — o caminho construído no [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html), que a [seção 11.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/02-usando-gradiente-descendente.html) já usou para chegar ao mesmo lugar que a fórmula fechada.

### O erro e o seu gradiente

A função de erro é quase idêntica à do capítulo anterior. A única mudança é que, em vez de esperar dois parâmetros `[alpha, beta]`, ela recebe um vetor de tamanho arbitrário:

In [ ]:
from typing import List
from scratch.linear_algebra import dot, Vector

def predict(x: Vector, beta: Vector) -> float:
    """assume que o primeiro elemento de x é 1"""
    return dot(x, beta)

def error(x: Vector, y: float, beta: Vector) -> float:
    return predict(x, beta) - y

def squared_error(x: Vector, y: float, beta: Vector) -> float:
    return error(x, y, beta) ** 2

x = [1, 2, 3]
y = 30
beta = [4, 4, 4]  # previsão = 4 + 8 + 12 = 24

assert error(x, y, beta) == -6
assert squared_error(x, y, beta) == 36

O gradiente é onde a generalização compensa. No capítulo anterior, as duas derivadas parciais foram escritas à mão, uma de cada vez. Aqui elas viram uma única compreensão de lista:

In [ ]:
def sqerror_gradient(x: Vector, y: float, beta: Vector) -> Vector:
    err = error(x, y, beta)
    return [2 * err * x_i for x_i in x]

assert sqerror_gradient(x, y, beta) == [-12, -24, -36]

> **🟩 Exemplo**
>
> Vale conferir a conta. A perda de um ponto é $(\,\mathbf{x} \cdot \beta - y\,)^2$. Derivando em relação a $\beta_j$ pela regra da cadeia: a derivada externa dá $2(\mathbf{x} \cdot \beta - y)$, e a derivada interna de $\mathbf{x} \cdot \beta$ em relação a $\beta_j$ é simplesmente $x_j$ — porque $\beta_j$ aparece uma única vez naquela soma, multiplicando $x_j$.
>
> $$
> \frac{\partial}{\partial \beta_j} \left( \mathbf{x} \cdot \beta - y \right)^2 = 2 \left( \mathbf{x} \cdot \beta - y \right) x_j
> $$
>
> Ou seja: `2 * err * x_j`, para cada `j`. É exatamente o que a compreensão de lista escreve — e repare que ela vale para o termo constante também, porque ali $x_0 = 1$ e a derivada dá `2 * err`, sem caso especial nenhum. É a coluna de 1 da [seção 12.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/01-o-modelo.html) pagando dividendo.

### O ajustador

Com o gradiente pronto, `least_squares_fit` é o gradiente descendente em minibatch da [seção 5.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/06-minibatch-e-estocastico.html), aplicado a qualquer conjunto de dados:

In [ ]:
import random
import tqdm
from scratch.linear_algebra import vector_mean
from scratch.gradient_descent import gradient_step

def least_squares_fit(xs: List[Vector],
                      ys: List[float],
                      learning_rate: float = 0.001,
                      num_steps: int = 1000,
                      batch_size: int = 1) -> Vector:
    """
    Encontra o beta que minimiza a soma dos erros ao quadrado,
    supondo o modelo y = dot(x, beta).
    """
    # Começa com um chute aleatório
    guess = [random.random() for _ in xs[0]]

    for _ in tqdm.trange(num_steps, desc="least squares fit"):
        for start in range(0, len(xs), batch_size):
            batch_xs = xs[start:start+batch_size]
            batch_ys = ys[start:start+batch_size]

            gradient = vector_mean([sqerror_gradient(x, y, guess)
                                    for x, y in zip(batch_xs, batch_ys)])
            guess = gradient_step(guess, gradient, -learning_rate)

    return guess

Não há nada de novo aqui, e esse é o ponto. `gradient_step` é a mesma função de cinco linhas do [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/03-usando-o-gradiente.html); `vector_mean` é do [Capítulo 4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/01-vetores.html); a estrutura de laço externo sobre passos e laço interno sobre lotes é a da [seção 5.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/06-minibatch-e-estocastico.html). Passar de uma variável explicativa para três não exigiu **nenhuma** mudança no otimizador — só que o gradiente devolvesse uma lista mais comprida.

### Ajustando aos dados

In [ ]:
import random
random.seed(0)
from scratch.multiple_regression import inputs
from scratch.statistics import daily_minutes_good
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
random.seed(0)
learning_rate = 0.001

beta = least_squares_fit(inputs, daily_minutes_good, learning_rate, 5000, 25)

assert 30.50 < beta[0] < 30.70  # constante
assert  0.96 < beta[1] <  1.00  # número de amigos
assert -1.89 < beta[2] < -1.85  # horas de trabalho por dia
assert  0.91 < beta[3] <  0.94  # tem doutorado

beta

Ou seja, o modelo ajustado é:

$$
\text{minutos} = 30{,}51 + 0{,}975 \,\text{amigos} - 1{,}851 \,\text{horas de trabalho} + 0{,}914 \,\text{doutorado}
$$

Os hiperparâmetros — taxa de aprendizado 0,001, 5.000 passos, lotes de 25 — foram escolhidos por tentativa e erro, e o próprio Grus diz isso no livro. Não há fórmula para eles; há o diagnóstico da [seção 5.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/04-escolhendo-o-tamanho-do-passo.html) para quando o passo é grande demais e a paciência para quando é pequeno demais.

> **❗ Importante — Estes quatro números são uma aproximação**
>
> O mínimo exato da soma dos erros ao quadrado, para estes dados, é
>
> $$
> \text{minutos} = 30{,}579 + 0{,}9725 \,\text{amigos} - 1{,}865 \,\text{horas de trabalho} + 0{,}9232 \,\text{doutorado}
> $$
>
> O nosso ajuste parou em 30,51, 0,975, −1,851 e 0,914 — diferenças que vão da terceira casa (0,9725 contra 0,975) à segunda (−1,865 contra −1,851).
>
> A explicação não é semente infeliz nem passos de menos. Gradiente descendente **estocástico**, com taxa de aprendizado fixa e ordem de lote fixa, não pousa no mínimo: ele entra numa órbita em torno dele, como a [seção 12.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/02-hipoteses-do-minimos-quadrados.html) mostrou rodando. Mais passos percorrem a mesma órbita de novo. A distância que sobra é do método, não da paciência.
>
> Mas repare no que a existência daquele mínimo exato significa: **a regressão múltipla tem, sim, solução fechada.** Ela não desapareceu quando saímos de uma variável para três. O que aconteceu é que ela deixou de caber nas duas médias, na correlação e nos dois desvios padrão do [Capítulo 11](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/01-o-modelo.html), e passou a exigir resolver um sistema linear — o que, por sua vez, exigiria construir do zero maquinaria de álgebra linear que este livro não construiu. É uma decisão de escopo, não uma impossibilidade matemática.
>
> Vale ser literal, então, sobre o que a [seção 11.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/02-usando-gradiente-descendente.html) prometeu no fim: quando aquele texto diz que aqui "a álgebra ainda fecha, mas fica pesada", ele está dizendo exatamente isto. Fecha mesmo. Os modelos que **não** têm fórmula fechada nenhuma começam no [Capítulo 13](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/index.html), com a regressão logística.

> **💡 Dica — Na prática: `scikit-learn`**
>
> ```python
> from sklearn.linear_model import LinearRegression
>
> X = [x[1:] for x in inputs]     # sem a coluna de 1
> modelo = LinearRegression().fit(X, daily_minutes_good)
>
> modelo.intercept_, modelo.coef_
> ```
>
> Isso devolve, direto, os coeficientes exatos — os 30,579 / 0,9725 / −1,865 / 0,9232 citados acima, sem semente, sem taxa de aprendizado e sem barra de progresso. `LinearRegression` não itera: ela resolve o sistema de mínimos quadrados por decomposição em valores singulares, que é a álgebra linear fora do escopo mencionada acima, implementada em LAPACK.
>
> Duas coisas que vale saber, agora que você viu o código por dentro:
>
> - **A biblioteca é exata e o nosso código é aproximado**, e isso não é um detalhe de qualidade — é a diferença entre resolver um sistema e descer uma encosta. Para regressão linear, resolver o sistema é sempre melhor. Escrevemos o gradiente descendente aqui porque ele é o método que **continua funcionando** quando a fórmula fechada desaparece, o que acontece já no próximo capítulo.
> - **Quando os dados são grandes demais para caber na memória**, a decomposição deixa de ser viável e o gradiente descendente volta a ser a opção prática. O `scikit-learn` tem `SGDRegressor` para exatamente isso — e ele pede `learning_rate`, `max_iter` e `random_state`, os mesmos três hiperparâmetros que o nosso `least_squares_fit` pede.

## Interpretando o Modelo

> **📌 Nota**
>
> Esta seção corresponde a *Interpreting the Model*, do capítulo 15 de Grus (2019).

A [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/03-ajustando-o-modelo.html) produziu quatro números. Ajustar é a parte fácil; dizer o que eles significam é onde a estatística aplicada costuma errar — e o erro não é de conta, é de leitura.

In [ ]:
import random
random.seed(0)
from scratch.multiple_regression import inputs, least_squares_fit
from scratch.statistics import daily_minutes_good
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
random.seed(0)
beta = least_squares_fit(inputs, daily_minutes_good, 0.001, 5000, 25)

In [ ]:
for nome, coef in zip(["constante", "amigos", "horas de trabalho", "doutorado"], beta):
    print(f"{nome:20s} {coef:8.4f}")

A leitura padrão desses números é: cada coeficiente é o efeito da sua variável **mantendo todo o resto constante**. Nas palavras que se usam numa reunião:

- Mantendo tudo o mais constante, cada amigo a mais corresponde a cerca de **um minuto a mais** por dia no site.
- Mantendo tudo o mais constante, cada hora a mais de trabalho por dia corresponde a cerca de **dois minutos a menos** por dia no site.
- Mantendo tudo o mais constante, ter doutorado corresponde a cerca de **um minuto a mais** por dia no site.

Essa leitura está correta. Ela também contém, na primeira parte de cada frase, a expressão mais perigosa da estatística aplicada.

### "Mantendo tudo o mais constante"

> **❗ Importante**
>
> O coeficiente 0,975 **não** é o efeito de ter mais um amigo. É o efeito de ter mais um amigo **entre pessoas com as mesmas horas de trabalho e o mesmo status de doutorado**.
>
> São afirmações diferentes, e a diferença tem consequência. A [seção 12.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/02-hipoteses-do-minimos-quadrados.html) mostrou o mesmo coeficiente valendo 0,904 no modelo de uma variável e 0,973 no modelo de três. Nenhum dos dois está errado: eles respondem a perguntas distintas. O primeiro responde *"quanto mais tempo, em média, passa no site quem tem um amigo a mais?"*. O segundo responde *"quanto mais tempo passa no site quem tem um amigo a mais, comparado com alguém que trabalha as mesmas horas e tem a mesma formação?"*. Se essas duas perguntas têm respostas diferentes, é porque o número de amigos anda junto com as outras variáveis — e é isso que os dados dizem.

O problema é que "mantendo tudo o mais constante" descreve uma comparação hipotética, e nada garante que ela seja uma comparação **que os dados sustentam**. A [seção 12.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/02-hipoteses-do-minimos-quadrados.html) já mediu isso neste conjunto: abaixo de 6 amigos o usuário tem doutorado, acima de 6 não tem, e a única faixa em que os dois grupos coexistem é a de exatamente 6 amigos.

O coeficiente de `doutorado` afirma comparar duas pessoas idênticas em tudo, exceto o doutorado. Nos dados, essa comparação só é observável em 22 dos 203 usuários — os que têm exatamente 6 amigos. Para todos os outros, "mesma quantidade de amigos, doutorado diferente" **não é um caso raro: é um caso inexistente**. O modelo produz o coeficiente mesmo assim, porque a álgebra não sabe disso; ele o produz extrapolando.

> **🔷 Conceito**
>
> Um coeficiente de regressão múltipla é uma **comparação condicional**, e ela só significa alguma coisa na região do espaço de variáveis onde os dados de fato variam de forma independente. Onde duas variáveis andam coladas, "manter uma constante enquanto a outra varia" descreve uma situação que o conjunto de dados nunca mostrou.
>
> O modelo não avisa. Ele devolve um número com quatro casas decimais, e o número parece tão sólido quanto os outros. A [seção 12.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/07-erros-padrao-dos-coeficientes.html) mostra como descobrir que ele não é.

> **⚠️ Atenção — E a causalidade, de novo**
>
> O [Capítulo 11](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/01-o-modelo.html) já avisou que correlação não estabelece causa, e que assumir "mais amigos **causam** mais tempo no site" era uma suposição trazida de fora dos dados. Vale registrar por que a regressão múltipla **não** melhora essa situação tanto quanto parece.
>
> O argumento tentador é: "controlei pelas horas de trabalho e pelo doutorado, então o que sobrou é o efeito puro dos amigos". Ele não se sustenta, e o motivo é a segunda hipótese da [seção 12.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/02-hipoteses-do-minimos-quadrados.html): controlar pelas variáveis que você **tem** não diz nada sobre as que você **não tem**. Se existe um fator não medido — tempo livre, idade, tipo de trabalho — que afeta tanto o número de amigos quanto o tempo no site, o coeficiente de `amigos` continua absorvendo esse efeito, com três variáveis de controle exatamente como absorvia com nenhuma.
>
> Regressão múltipla melhora **previsão**. Ela só melhora **inferência causal** quando existe um argumento, vindo de fora dos dados, de que as variáveis omitidas não importam. Esse argumento nunca sai da conta.

### O que o modelo não captura

O modelo diz que o efeito de cada variável é o mesmo para todo mundo, e que ele é constante ao longo de toda a faixa de valores. Nenhuma das duas coisas é obrigatoriamente verdade.

**Interações.** É possível que o efeito das horas de trabalho seja diferente para quem tem muitos amigos e para quem tem poucos. O modelo atual não tem como expressar isso: `beta[2]` é um número só. O jeito de capturar essa possibilidade é acrescentar uma variável que seja o **produto** de `amigos` por `horas`. O coeficiente dessa nova coluna deixa o efeito de `horas` crescer (ou encolher) conforme o número de amigos aumenta — e continua sendo um modelo linear, porque ele é linear **nos parâmetros**, não nas variáveis originais.

**Não linearidades.** É possível que mais amigos aumentem o tempo no site **até certo ponto**, e que a partir dali amigos demais tornem a experiência sobrecarregada e o tempo caia. Uma reta não faz curva; uma reta somada a uma coluna com o **quadrado** de `amigos` faz. Mesmo truque, mesma observação: o modelo continua linear nos parâmetros.

Nos dois casos, note **quem** faz o trabalho: você. É preciso desconfiar da interação, escrever a coluna do produto, desconfiar da curva, escrever a coluna do quadrado. Modelos que capturam interações e não linearidades sem que ninguém as escreva à mão existem, e este livro chega a eles: as árvores de decisão do [Capítulo 14](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/index.html) partem os dados em regiões e por isso representam interações de graça, e as redes neurais do [Capítulo 15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html) compõem transformações não lineares umas sobre as outras. O que eles cobram por isso é justamente o que esta seção acabou de fazer: a leitura direta de um coeficiente.

Repare que os dois casos usam a mesma ferramenta: construir uma coluna nova a partir das que já existem, e deixar a regressão descobrir o coeficiente dela. Isso é **engenharia de atributos**, o assunto da [seção 8.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/06-extracao-e-selecao-de-atributos.html), aplicado aqui de forma bem literal.

E é aqui que o capítulo vira. Não há limite para o número de produtos, logaritmos, quadrados e potências que podemos acrescentar — e cada um deles é uma coluna nova, com um coeficiente novo, que o ajuste vai estimar de bom grado. A partir do momento em que começamos a inventar variáveis, precisamos de um jeito de saber quais delas **importam**.

A [seção 12.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/05-qualidade-do-ajuste.html) mostra que a métrica que temos até agora — o R² — é exatamente a ferramenta errada para essa pergunta.

> **💡 Dica — Na prática: `scikit-learn`**
>
> Construir colunas de produtos e potências à mão fica tedioso rápido, e o `scikit-learn` automatiza isso:
>
> ```python
> from sklearn.preprocessing import PolynomialFeatures
> from sklearn.pipeline import make_pipeline
> from sklearn.linear_model import LinearRegression
>
> modelo = make_pipeline(PolynomialFeatures(degree=2, include_bias=False),
>                        LinearRegression())
> modelo.fit(X, y)
> ```
>
> `PolynomialFeatures(degree=2)` recebe as três colunas originais e devolve nove: as três originais, os três quadrados e os três produtos par a par. Note o `include_bias=False` — por padrão essa transformação também acrescenta a coluna de 1, que somada ao `fit_intercept=True` do `LinearRegression` produziria a colinearidade exata da [seção 12.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/02-hipoteses-do-minimos-quadrados.html).
>
> Isso ilustra bem o que a biblioteca esconde e o que ela não esconde. Ela esconde a digitação: gerar nove colunas com uma linha. Ela **não** esconde o julgamento: `degree=3` sobre 10 variáveis produz 285 colunas, o ajuste roda sem reclamar, o R² sobe — e você fica com um modelo que decorou o conjunto de treino. É o sobreajuste do [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/03-overfitting-e-underfitting.html), e a única defesa automática que existe é a da [seção 12.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/08-regularizacao.html).

## Qualidade do Ajuste

> **📌 Nota**
>
> Esta seção corresponde a *Goodness of Fit*, do capítulo 15 de Grus (2019).

O [Capítulo 11](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/01-o-modelo.html) mediu a qualidade do ajuste com o **coeficiente de determinação**, o R²: a fração da variação total em `y` que o modelo captura. A generalização para várias variáveis não muda nada na ideia; só troca a assinatura da função:

In [ ]:
import random
random.seed(0)
from scratch.multiple_regression import inputs, least_squares_fit
from scratch.statistics import daily_minutes_good
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
from typing import List
from scratch.linear_algebra import Vector, dot
from scratch.simple_linear_regression import total_sum_of_squares

def error(x: Vector, y: float, beta: Vector) -> float:
    return dot(x, beta) - y

def multiple_r_squared(xs: List[Vector], ys: Vector, beta: Vector) -> float:
    sum_of_squared_errors = sum(error(x, y, beta) ** 2
                                for x, y in zip(xs, ys))
    return 1.0 - sum_of_squared_errors / total_sum_of_squares(ys)

`total_sum_of_squares` é a mesma função do capítulo anterior: a variação quadrática total dos `y` em torno da própria média, que é o erro do modelo mais burro possível — aquele que ignora as variáveis explicativas e sempre chuta a média.

In [ ]:
random.seed(0)
beta = least_squares_fit(inputs, daily_minutes_good, 0.001, 5000, 25)

multiple_r_squared(inputs, daily_minutes_good, beta)

O R² subiu de **0,329**, no modelo de uma variável do Capítulo 11, para **0,680**. Duas variáveis a mais, e a fração da variação explicada praticamente dobrou. Esse é o resultado que se leva para a reunião.

E é exatamente onde este capítulo precisa parar e desconfiar.

### O R² nunca desce

> **❗ Importante**
>
> Acrescentar uma variável a uma regressão **necessariamente** não diminui o R². Nunca. Não importa qual seja a variável.
>
> O argumento é curto e não tem brecha. O modelo de uma variável é um **caso particular** do modelo de três: basta que os coeficientes de `horas de trabalho` e `doutorado` valham zero. Ou seja, aquele modelo estava entre os candidatos que o ajuste de três variáveis considerou — e foi rejeitado em favor de outro. O melhor modelo de três variáveis tem, portanto, soma de erros ao quadrado **no máximo** igual à do melhor modelo de uma. Como o R² é 1 menos essa soma dividida por uma constante, ele só pode ter subido, ou ficado igual.

O mesmo argumento vale para qualquer coluna acrescentada, e é aí que ele incomoda: ele vale também para uma coluna de números aleatórios sem relação nenhuma com o que estamos tentando prever. Zero é um coeficiente possível para ela, então o ajuste com a coluna de ruído nunca fica pior que o ajuste sem. Se o ajuste consegue tirar até um fiapo de vantagem daquele ruído — e com dados finitos ele quase sempre consegue —, o R² sobe.

Isso não é um argumento de papel. Dá para rodar:

In [ ]:
random.seed(1)
# vinte colunas de puro ruído, na mesma faixa das horas de trabalho
ruido = [[random.uniform(0, 10) for _ in range(20)] for _ in inputs]

for k in [0, 1, 5, 10, 15, 20]:
    xs = [x + r[:k] for x, r in zip(inputs, ruido)]
    random.seed(0)
    beta_k = least_squares_fit(xs, daily_minutes_good, 0.001, 15000, 25)
    r2 = multiple_r_squared(xs, daily_minutes_good, beta_k)
    plural = "coluna" if k == 1 else "colunas"
    print(f"{k:2d} {plural:7s} de puro ruído → R² = {r2:.4f}")

> **⚠️ Atenção — O que acabou de acontecer**
>
> Vinte colunas de `random.uniform(0, 10)`. Nenhuma delas sabe coisa alguma sobre quantos minutos por dia um usuário passa no site — são números sorteados, gerados sem olhar para `daily_minutes_good` uma única vez. Ainda assim, o R² subiu de 0,68 para 0,71.
>
> Um analista que acrescentasse essas vinte colunas ao modelo e relatasse "o R² subiu, o modelo melhorou" estaria enganando a si mesmo e a quem o ouvisse. O modelo não melhorou em nada: ele apenas ganhou vinte botões a mais para girar, e usou-os para se ajustar ao ruído específico destes 203 usuários. Aplicado a usuários novos, esse modelo prevê **pior** que o de três variáveis, não melhor.
>
> Este é o sobreajuste do [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/03-overfitting-e-underfitting.html) visto de um ângulo novo, e é a razão pela qual aquele capítulo insistiu em medir desempenho em dados que o modelo não viu. Aqui estamos medindo tudo no conjunto de treino, e o resultado é uma métrica que só sabe aplaudir.

> **📌 Nota — Um detalhe honesto sobre o experimento acima**
>
> O argumento da caixa anterior é sobre o **mínimo exato** da soma dos erros ao quadrado, e o nosso `least_squares_fit` não encontra o mínimo exato — ele dá 15.000 passos e para perto dele. Cabe perguntar, então, se o que a tabela mostra é o efeito das colunas de ruído ou a imprecisão do otimizador.
>
> É o efeito das colunas, e por uma margem confortável. Olhe a **segunda** linha da tabela: uma única coluna de ruído já levou o R² de 0,6800 para 0,6825. São uns poucos milésimos — a ordem de grandeza que a regra de bolso para "uma variável a mais" prevê com 203 pontos —, e são milésimos que **sobem**. A imprecisão do otimizador, medida contra a solução exata do mesmo modelo de três variáveis, é de 0,000026 (0,679985 contra 0,680011): duas ordens de grandeza abaixo. Uma coluna sozinha já demonstra o ponto; as vinte apenas o tornam impossível de ignorar.
>
> Vale generalizar a lição, porque ela reaparece toda vez que se mede alguma coisa: **um efeito só é demonstrável quando ele é maior que a imprecisão do instrumento que o mede.** Quando os dois têm o mesmo tamanho, a medição não distingue "o efeito não existe" de "o efeito existe e eu não consigo vê-lo" — e concluir a primeira coisa é um erro. É a mesma disciplina que o [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html) pediu ao separar treino de teste, e é exatamente o que a [seção 12.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/07-erros-padrao-dos-coeficientes.html) vai fazer com cada coeficiente do modelo.

### O buraco que isso abre

O R² é uma métrica útil e continuará sendo. O que ele **não** consegue fazer é responder à pergunta que a [seção 12.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/04-interpretando-o-modelo.html) deixou em aberto: dado que podemos inventar quantas colunas quisermos — produtos, quadrados, logaritmos —, quais delas merecem estar no modelo?

Ele não consegue porque a resposta dele é sempre a mesma: acrescente. Uma métrica que nunca piora não pode servir de critério para deixar algo de fora.

Precisamos de outra coisa. Precisamos, para cada coeficiente, de uma medida de **quanto confiar nele** — se aquele 0,914 do `doutorado` é um efeito real ou um número que teria saído completamente diferente se tivéssemos coletado outros 203 usuários. Essa medida se chama **erro padrão** do coeficiente.

> **❗ Importante**
>
> A abordagem clássica para calcular erros padrão parte de uma hipótese adicional: a de que os erros $\varepsilon_i$ são variáveis normais independentes, com média 0 e um desvio padrão $\sigma$ comum e desconhecido — a mesma hipótese que a [seção 11.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/03-maxima-verossimilhanca.html) usou para justificar mínimos quadrados. Sob ela, existe uma fórmula, e ela sai da mesma álgebra linear que a [seção 12.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/03-ajustando-o-modelo.html) disse estar fora do escopo deste livro.
>
> Ficaríamos, então, sem erros padrão — a não ser que exista um caminho que não passe por álgebra linear nenhuma. Existe, e ele não passa nem por teoria estatística. Ele passa por reamostrar os próprios dados, e é o assunto da [próxima seção](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/06-digressao-o-bootstrap.html).

> **💡 Dica — Na prática: `scikit-learn`**
>
> `modelo.score(X, y)` devolve exatamente este R², e `sklearn.metrics.r2_score(y_verdadeiro, y_previsto)` faz a mesma conta a partir de duas listas. Nenhum dos dois protege você do que esta seção mostrou: os dois sobem quando você acrescenta ruído.
>
> O que se usa de verdade para escolher variáveis:
>
> - **R² ajustado**, que penaliza o número de parâmetros — ele *pode* descer quando você acrescenta uma variável inútil. O `scikit-learn` não o oferece; o `statsmodels` traz na tabela de resumo. É uma correção de fórmula, barata e limitada.
> - **Validação cruzada** (`sklearn.model_selection.cross_val_score`), que mede o R² em dados que o modelo não viu no ajuste. É a resposta certa para a pergunta "este modelo prevê melhor?", e é a generalização direta da divisão treino/teste do [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/04-correcao.html). Se você levar só uma ferramenta desta seção, leve esta.
> - **Erros padrão e valores-p**, que respondem a uma pergunta diferente e complementar: não "este modelo prevê melhor?", mas "este coeficiente específico é distinguível de zero?". São o assunto das seções [12.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/06-digressao-o-bootstrap.html) e [12.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/07-erros-padrao-dos-coeficientes.html), e o `scikit-learn` **não os calcula** — para eles, o `statsmodels` é a ferramenta.

## Digressão: O Bootstrap

> **📌 Nota**
>
> Esta seção corresponde a *Digression: The Bootstrap*, do capítulo 15 de Grus (2019).

A [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/05-qualidade-do-ajuste.html) terminou precisando de uma coisa que não tínhamos como calcular: uma medida de quanto confiar em cada coeficiente. O título do Grus chama o que vem agora de digressão. É a ideia mais reaproveitável do capítulo inteiro — ela sai da regressão por completo, e é exatamente por isso que serve para tudo.

O problema, no geral: você tem uma amostra de $n$ pontos, vinda de alguma distribuição que você desconhece.

```python
data = get_sample(num_points=n)
```

`get_sample` é imaginária: ela não existe neste livro nem em biblioteca nenhuma, e está aí só para dizer "de algum lugar veio uma amostra". O ponto do bootstrap é justamente que, na vida real, você quase nunca pode chamá-la de novo.

Você calcula uma estatística sobre ela — a mediana, digamos — e a usa como estimativa da mediana da distribuição verdadeira. A pergunta é: **quanta confiança essa estimativa merece?**

### Duas amostras com a mesma mediana

Se todos os pontos da amostra estiverem muito perto de 100, é razoável acreditar que a mediana verdadeira está perto de 100. Mas se metade dos pontos estiver perto de 0 e a outra metade perto de 200, a mediana da amostra também dá perto de 100 — e agora a confiança deveria ser bem menor.

In [ ]:
import random
from scratch.statistics import median, standard_deviation
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
random.seed(0)

# 101 pontos, todos bem perto de 100
close_to_100 = [99.5 + random.random() for _ in range(101)]

# 101 pontos, 50 deles perto de 0 e 50 deles perto de 200
far_from_100 = ([99.5 + random.random()] +
                [random.random() for _ in range(50)] +
                [200 + random.random() for _ in range(50)])

median(close_to_100), median(far_from_100)

As duas medianas dão praticamente 100. Nenhuma estatística calculada sobre a amostra, sozinha, distingue os dois casos — e no entanto os dois casos são radicalmente diferentes.

### A ideia

Se pudéssemos coletar amostras novas repetidamente, o problema estaria resolvido: bastaria calcular a mediana de cada uma e olhar a dispersão dessas medianas. Uma dispersão pequena significaria uma estimativa confiável.

Quase nunca podemos. Coletar dados custa caro, e frequentemente a amostra que temos é toda a que vamos ter.

> **🔷 Conceito**
>
> O **bootstrap** faz o seguinte salto: em vez de coletar amostras novas da população, sorteia amostras novas **da própria amostra**, com reposição, cada uma do mesmo tamanho do original. Calcula a estatística em cada uma delas, e olha a dispersão dos resultados.
>
> A justificativa intuitiva é que a amostra que você tem é a melhor estimativa disponível da população de onde ela veio. Reamostrar dela imita, de forma barata, o processo de coletar dados novos.
>
> O que torna a ideia poderosa é o que ela **não** exige: nenhuma suposição sobre a forma da distribuição, nenhuma fórmula específica para a estatística, nenhuma teoria. A estatística pode ser a mediana, a correlação, o coeficiente de uma regressão, o percentil 90 ou qualquer coisa que você consiga escrever como função de uma lista. O bootstrap trata a estatística como caixa-preta — o que é irônico, num livro dedicado a abri-las, e é exatamente por isso que ele funciona em tantos lugares.

O código todo são duas funções, com um `return` cada:

In [ ]:
from typing import TypeVar, Callable, List

X = TypeVar('X')        # tipo genérico para os dados
Stat = TypeVar('Stat')  # tipo genérico para a "estatística"

def bootstrap_sample(data: List[X]) -> List[X]:
    """sorteia len(data) elementos com reposição"""
    return [random.choice(data) for _ in data]

def bootstrap_statistic(data: List[X],
                        stats_fn: Callable[[List[X]], Stat],
                        num_samples: int) -> List[Stat]:
    """avalia stats_fn sobre num_samples amostras bootstrap de data"""
    return [stats_fn(bootstrap_sample(data)) for _ in range(num_samples)]

Note que `stats_fn` é um parâmetro. `bootstrap_statistic` não sabe nem se importa com o que está sendo calculado — e é por isso que a mesma função vai servir, sem uma linha de mudança, para estimar erros padrão de regressão na [seção 12.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/07-erros-padrao-dos-coeficientes.html).

### Aplicando aos dois conjuntos

In [ ]:
random.seed(0)

medians_close = bootstrap_statistic(close_to_100, median, 100)
medians_far   = bootstrap_statistic(far_from_100, median, 100)

assert standard_deviation(medians_close) < 1
assert standard_deviation(medians_far)   > 90

standard_deviation(medians_close), standard_deviation(medians_far)

O desvio padrão das 100 medianas do primeiro conjunto é de cerca de **0,038**. O do segundo é de cerca de **97**. A diferença é de mais de três ordens de grandeza — e nenhuma delas apareceria olhando apenas para as duas medianas originais, que davam as duas perto de 100.

In [ ]:
# Figura: As 100 medianas bootstrap de cada conjunto. Mesma escala nos dois painéis.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)

ax1.hist(medians_close, bins=20, range=(-10, 210), color='tab:blue')
ax1.set_title("perto de 100")
ax1.set_xlabel("mediana da amostra bootstrap")
ax1.set_ylabel("frequência")

ax2.hist(medians_far, bins=20, range=(-10, 210), color='tab:red')
ax2.set_title("longe de 100")
ax2.set_xlabel("mediana da amostra bootstrap")

for ax in (ax1, ax2):
    ax.axvline(100, color='black', linestyle='--', linewidth=1)

plt.tight_layout()
plt.show()

À esquerda, as 100 medianas caem todas numa única barra em torno de 100 — a estimativa é estável. À direita, elas se espalham em três grupos: um perto de 0, um perto de 200 e um punhado no meio. Contando:

In [ ]:
perto_de_zero = sum(1 for m in medians_far if m < 50)
perto_de_200  = sum(1 for m in medians_far if m > 150)

print(f"medianas perto de 0:   {perto_de_zero}")
print(f"medianas perto de 200: {perto_de_200}")
print(f"medianas no meio:      {100 - perto_de_zero - perto_de_200}")

Isso é o bootstrap dizendo, alto e claro, que a mediana daquele segundo conjunto não significa nada. O valor 100 que ela devolveu é um artefato de haver exatamente um ponto no meio: mude um único elemento da amostra e a mediana pula para 0 ou para 200.

> **⚠️ Atenção**
>
> Este exemplo é extremo de propósito, e nele daria para perceber o problema simplesmente olhando os dados. Em geral não dá — e o valor do bootstrap é justamente funcionar quando a inspeção visual não funciona: em muitas dimensões, com estatísticas complicadas, sobre dados que ninguém consegue desenhar.
>
> Duas ressalvas honestas, porque o bootstrap não é mágica:
>
> - Ele mede a incerteza **devida à amostragem**, e só ela. Se a sua amostra for enviesada — coletada de um jeito que não representa a população —, o bootstrap vai reamostrar esse viés com toda a fidelidade e devolver uma estimativa de incerteza pequena e confiante em torno da resposta errada.
> - Ele custa `num_samples` vezes o custo de calcular a estatística. Para uma mediana isso não é nada. Para um ajuste de regressão que leva quase um segundo, cem amostras levam mais de um minuto — e é exatamente o que a próxima seção vai pagar.

> **💡 Dica — Na prática: `scipy`**
>
> ```python
> from scipy.stats import bootstrap
> import numpy as np
>
> resultado = bootstrap((np.array(far_from_100),), np.median,
>                       n_resamples=10_000, random_state=0)
>
> resultado.standard_error
> resultado.confidence_interval
> ```
>
> `scipy.stats.bootstrap` faz o que `bootstrap_statistic` faz, com três diferenças que valem entender agora que você viu o miolo:
>
> - Ela devolve **intervalos de confiança** já corrigidos (o método `BCa`, padrão, ajusta viés e assimetria da distribuição bootstrap). Nós devolvemos a lista crua de estatísticas e calculamos o desvio padrão à mão.
> - Ela reamostra de forma vetorizada, o que torna 10.000 reamostragens viável onde as nossas 100 já custam.
> - Ela aceita várias amostras de uma vez, para estatísticas de dois grupos (diferença de médias, por exemplo).
>
> O `scikit-learn` também tem `sklearn.utils.resample`, que é essencialmente o nosso `bootstrap_sample` — reamostragem com reposição, sem nada em volta. E o mesmo mecanismo de reamostrar-com-reposição reaparece, com outro objetivo, no **bagging** que sustenta as florestas aleatórias do [Capítulo 14](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/06-florestas-aleatorias.html): lá não se mede incerteza, se treina uma árvore em cada reamostra e se combinam as previsões.

## Erros Padrão dos Coeficientes

> **📌 Nota**
>
> Esta seção corresponde a *Standard Errors of Regression Coefficients*, do capítulo 15 de Grus (2019).

A [seção 12.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/05-qualidade-do-ajuste.html) precisava saber quanto confiar em cada coeficiente, e a via clássica exigia álgebra linear que este livro não construiu. A [seção 12.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/06-digressao-o-bootstrap.html) trouxe uma alternativa que não exige teoria nenhuma. Agora é só apontar uma para a outra.

A ideia é literalmente a mesma da seção anterior, com `beta` no lugar da mediana: reamostramos os dados com reposição, reajustamos o modelo em cada reamostra e olhamos como cada coeficiente varia. Se o coeficiente de `num_amigos` mal se move entre as reamostras, temos boa razão para confiar nele. Se ele oscila muito, não temos.

### Reamostrando pares

Há uma única sutileza técnica. Não podemos reamostrar `inputs` e `daily_minutes_good` separadamente — isso embaralharia quais minutos pertencem a qual usuário. Precisamos reamostrar **pares** $(x_i, y_i)$ e desmontá-los depois:

In [ ]:
import random
random.seed(0)
from scratch.multiple_regression import (
    inputs, least_squares_fit, bootstrap_statistic,
)
from scratch.statistics import daily_minutes_good, standard_deviation
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
from typing import List, Tuple
from scratch.linear_algebra import Vector

learning_rate = 0.001

def estimate_sample_beta(pairs: List[Tuple[Vector, float]]) -> Vector:
    x_sample = [x for x, _ in pairs]
    y_sample = [y for _, y in pairs]
    return least_squares_fit(x_sample, y_sample, learning_rate, 5000, 25)

E então é uma chamada só:

In [ ]:
random.seed(0)   # para você obter os mesmos resultados

# Isto leva mais de um minuto!
bootstrap_betas = bootstrap_statistic(list(zip(inputs, daily_minutes_good)),
                                      estimate_sample_beta,
                                      100)

len(bootstrap_betas), bootstrap_betas[0]

> **📌 Nota**
>
> Cem ajustes completos, cada um com 5.000 passos de gradiente descendente sobre 203 pontos, em Python puro. Isso leva cerca de **90 segundos**.
>
> O custo é o preço do método, não um defeito da nossa implementação: o bootstrap **é** repetir a estimativa cem vezes. O próprio Grus (2019) observa que estimativas melhores viriam de mais amostras e mais iterações por amostra, "mas não temos o dia inteiro".
>
> Vale ver o custo como informação, não como incômodo. Quando a estatística é barata — a mediana da seção anterior —, o bootstrap é praticamente gratuito e não há razão para não usá-lo. Quando ela é cara, o bootstrap multiplica esse custo por cem, e aí a fórmula fechada da abordagem clássica deixa de ser um luxo teórico e passa a ser a diferença entre um segundo e uma tarde.

Com os 100 vetores em mãos, o erro padrão de cada coeficiente é o desvio padrão daquela coordenada:

In [ ]:
bootstrap_standard_errors = [
    standard_deviation([beta[i] for beta in bootstrap_betas])
    for i in range(4)]

bootstrap_standard_errors

In [ ]:
random.seed(0)
beta = least_squares_fit(inputs, daily_minutes_good, learning_rate, 5000, 25)

nomes = ["constante", "amigos", "horas de trabalho", "doutorado"]
for nome, b, s in zip(nomes, beta, bootstrap_standard_errors):
    print(f"{nome:20s} {b:8.4f}   ± {s:.4f}")

Os números já dizem a história antes de qualquer teste. O coeficiente de `amigos` é 0,975 com erro padrão de 0,103 — quase dez vezes maior que a própria incerteza. O de `doutorado` é 0,914 com erro padrão de 1,249: **a incerteza é maior que a estimativa**.

In [ ]:
# Figura: Distribuição dos 100 coeficientes bootstrap. Mesma escala horizontal nos três painéis; a linha tracejada marca o zero.
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), sharex=True, sharey=True)

todos = [b[i] for b in bootstrap_betas for i in (1, 2, 3)]
faixa = (min(todos) - 0.3, max(todos) + 0.3)

for ax, i, titulo in zip(axes, [1, 2, 3],
                         ["amigos", "horas de trabalho", "doutorado"]):
    coluna = [b[i] for b in bootstrap_betas]
    ax.hist(coluna, bins=30, range=faixa, color='tab:blue', edgecolor='white')
    ax.axvline(0, color='black', linestyle='--', linewidth=1.2)
    ax.set_title(titulo, fontsize=10)
    ax.set_xlabel("coeficiente")

axes[0].set_ylabel("frequência")
plt.tight_layout()
plt.show()

O gráfico mostra o que a tabela resume, e a escala compartilhada é o que torna a comparação legítima: com um eixo por painel, três distribuições de larguras completamente diferentes pareceriam igualmente espalhadas. Na mesma régua, `amigos` e `horas de trabalho` se apertam em faixas estreitas, inteiramente de um lado do zero — nenhuma das 100 reamostras produziu um coeficiente de sinal trocado. A de `doutorado` ocupa quase toda a largura do gráfico e atravessa o zero: uma boa parte das reamostras produziu um coeficiente **negativo**.

In [ ]:
for i, nome in zip([1, 2, 3], ["amigos", "horas de trabalho", "doutorado"]):
    coluna = [b[i] for b in bootstrap_betas]
    negativos = sum(1 for v in coluna if v < 0)
    print(f"{nome:20s} de {min(coluna):7.3f} a {max(coluna):6.3f}   "
          f"negativos: {negativos:3d} de 100")

### De erro padrão a valor-p

O erro padrão permite testar a hipótese "$\beta_j$ é igual a 0". Sob essa hipótese nula — e sob as suposições sobre a distribuição de $\varepsilon_i$ —, a estatística

$$
t_j = \frac{\hat{\beta_j}}{\hat{\sigma_j}}
$$

(a estimativa do coeficiente dividida pela estimativa do erro padrão dele) segue uma distribuição *t* de Student com $n - k$ graus de liberdade.

Se você nunca encontrou a *t*: ela é uma parente da normal padrão, com o mesmo formato de sino, um pouco mais achatada e com caudas mais gordas — é o que sobra quando você divide por um desvio padrão que também foi **estimado** dos dados, em vez de conhecido. O quanto ela engorda depende dos **graus de liberdade**, que são o número de observações menos o número de parâmetros que você já gastou estimando ($n - k$): quanto mais pontos sobram por parâmetro, mais confiável é o desvio padrão estimado e menos a *t* se afasta da normal.

Não temos uma função `students_t_cdf`, e escrevê-la do zero seria uma digressão maior que este capítulo comporta. Mas conforme os graus de liberdade crescem, a *t* se aproxima cada vez mais de uma normal padrão — e com $n = 203$ e $k = 4$, essa aproximação é boa o bastante. Usamos a `normal_cdf` que já temos:

In [ ]:
from scratch.probability import normal_cdf

def p_value(beta_hat_j: float, sigma_hat_j: float) -> float:
    if beta_hat_j > 0:
        # se o coeficiente é positivo, precisamos calcular o dobro da
        # probabilidade de ver um valor ainda *maior*
        return 2 * (1 - normal_cdf(beta_hat_j / sigma_hat_j))
    else:
        # caso contrário, o dobro da probabilidade de ver um valor *menor*
        return 2 * normal_cdf(beta_hat_j / sigma_hat_j)

# os coeficientes abaixo são a solução exata do sistema de mínimos quadrados;
# os nossos, na tabela acima, vêm do gradiente descendente e diferem na
# segunda ou terceira casa. Os erros padrão são os nossos, do bootstrap.
assert p_value(30.58, 1.27)   < 0.001  # termo constante
assert p_value(0.972, 0.103)  < 0.001  # amigos
assert p_value(-1.865, 0.155) < 0.001  # horas de trabalho
assert p_value(0.923, 1.249)  > 0.4    # doutorado

In [ ]:
print(f"{'variável':<20}{'coef.':>9}{'erro padrão':>14}{'t':>9}{'valor-p':>10}")
for nome, b, s in zip(nomes, beta, bootstrap_standard_errors):
    print(f"{nome:<20}{b:9.4f}{s:14.4f}{b/s:9.3f}{p_value(b, s):10.4f}")

> **🔷 Conceito**
>
> O valor-p responde a: *se o coeficiente verdadeiro fosse exatamente zero, com que frequência o acaso da amostragem produziria uma estimativa tão distante de zero quanto a que eu obtive?*
>
> Para `amigos` e `horas de trabalho`, a resposta é "praticamente nunca" — esses efeitos são reais. Para `doutorado`, a resposta é "quase metade das vezes". Um coeficiente de 0,914 com erro padrão de 1,249 é perfeitamente compatível com um efeito verdadeiro de zero, e também com um efeito de $-1$ ou de $+3$. Não sabemos.

> **📌 Nota — A hipótese que o bootstrap não precisou fazer**
>
> Repare no que **não** foi preciso supor para chegar até aqui. A via clássica — a fórmula fechada que a [seção 12.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/05-qualidade-do-ajuste.html) mencionou e que o callout no fim desta seção mostra em uso — parte de erros normais, independentes e com **variância constante**: a dispersão em torno do modelo tem que ser a mesma para quem passa 10 minutos por dia no site e para quem passa 90. Essa última exigência tem nome — **homocedasticidade** — e a [seção 11.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/03-maxima-verossimilhanca.html) prometeu que voltaríamos a ela. É aqui.
>
> Quando ela não vale, o caso comum é o erro crescer junto com o valor previsto: o modelo acerta na faixa baixa e erra feio na alta. Nesse cenário, o ajuste de mínimos quadrados continua devolvendo coeficientes utilizáveis, mas os **erros padrão** da fórmula fechada saem errados — e não erram todos para o mesmo lado, o que impediria até de corrigir a leitura de cabeça. O que quebra, então, é exatamente a camada que esta seção construiu: o erro padrão, o *t* e o valor-p. Nada na tabela de saída avisa; um coeficiente pode aparecer com p < 0,001 só porque a fórmula supôs uma variância que os dados não têm.
>
> O bootstrap de pares não faz essa suposição. Ele reamostra $(x_i, y_i)$ juntos, então cada reamostra herda a estrutura de dispersão do conjunto original, seja ela qual for — a dispersão dos 100 `beta` reflete a variabilidade que os dados de fato têm, não a que um modelo de erros supôs que eles teriam. É o que se compra com os 90 segundos de CPU: hipóteses a menos.

### Por que justamente o doutorado

Este resultado não deveria surpreender quem leu a [seção 12.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/02-hipoteses-do-minimos-quadrados.html): `tem_doutorado` é quase determinado por `num_amigos`, e a comparação capaz de separar os dois efeitos — mesmo número de amigos, formação diferente — só existe nos 22 usuários daquela seção.

Isso é multicolinearidade quase perfeita, e o erro padrão de 1,249 é exatamente a forma como ela se manifesta. Cada reamostra bootstrap sorteia um punhado diferente daqueles 22 usuários, e o coeficiente balança junto.

> **❗ Importante**
>
> Repare no arco que se fechou até aqui. A [seção 12.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/02-hipoteses-do-minimos-quadrados.html) apontou uma propriedade estrutural dos dados. A [seção 12.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/04-interpretando-o-modelo.html) explicou por que ela torna a frase "mantendo tudo o mais constante" vazia para aquele coeficiente. A [seção 12.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/05-qualidade-do-ajuste.html) mostrou que o R² é cego para esse tipo de problema. E o bootstrap, sem saber nada de nada disso — sem teoria, sem álgebra, sem sequer olhar para a matriz de correlações —, mediu o efeito no número que interessa.
>
> Falta o passo que **age** sobre esse diagnóstico. Até agora, tudo o que sabemos fazer com o coeficiente do `doutorado` é olhar a tabela, ver o 1,249 e decidir à mão remover a variável — o que exige um humano, um critério e uma rodada a mais de ajuste. A [seção 12.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/08-regularizacao.html) muda o objetivo que o ajuste persegue, de modo que um coeficiente que os dados não sustentam encolha sozinho, sem que ninguém precise olhar tabela nenhuma.

> **⚠️ Atenção — O que um valor-p não é**
>
> Três leituras erradas, todas comuns:
>
> - **"p = 0,46 prova que o doutorado não tem efeito."** Não prova. Prova que estes dados não conseguem distinguir o efeito de zero, o que é diferente. Ausência de evidência não é evidência de ausência — e neste caso sabemos até *por que* os dados não conseguem: eles quase não contêm a comparação necessária.
> - **"p < 0,05, então o efeito é importante."** Significância estatística é sobre distinguir de zero, não sobre tamanho. Com dados suficientes, um efeito de 0,001 minuto por amigo sairia com p < 0,001 e não teria a menor relevância prática.
> - **"testei vinte variáveis e uma deu p < 0,05."** Esperado: testando vinte variáveis inúteis a 5%, sai em média uma "significativa" por puro acaso. É o mesmo mecanismo que a [seção 12.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/05-qualidade-do-ajuste.html) mostrou com as vinte colunas de ruído, agora vestido de teste de hipótese.
>
> Testes mais elaborados — "pelo menos um dos $\beta_j$ é diferente de zero", ou "$\beta_1$ é igual a $\beta_2$" — existem, e o instrumento se chama teste *F*. Ele está fora do escopo deste livro, mas fica o nome para quando você precisar procurá-lo.

> **💡 Dica — Na prática: `statsmodels`**
>
> O fecho desta seção **não** é o `scikit-learn`, e a razão é direta: o `scikit-learn` simplesmente não faz isto. `LinearRegression` devolve `coef_` e nada mais: nem erro padrão, nem valor-p, nem intervalo de confiança. Não é uma omissão; é uma decisão de projeto. A biblioteca é feita para prever, e para prever esses números não servem.
>
> Quem faz é o `statsmodels`, uma biblioteca de estatística — não de aprendizado de máquina — construída em torno justamente do que o `scikit-learn` deixa de fora. Para experimentar o código abaixo, instale-o com `pip install statsmodels`.
>
> ```python
> import statsmodels.api as sm
>
> modelo = sm.OLS(daily_minutes_good, inputs).fit()   # inputs JÁ tem a coluna de 1
> print(modelo.summary())
> ```
>
> `summary()` imprime, de uma vez, a tabela que nós montamos à mão: coeficiente, erro padrão, estatística *t*, valor-p e intervalo de confiança de 95% para cada variável — mais R², R² ajustado, estatística *F* e uma lista de avisos de diagnóstico. Repare que `sm.OLS` **espera** a coluna de 1 explícita, exatamente como o nosso código (há `sm.add_constant` para quem não a tem), e resolve por álgebra linear em vez de gradiente descendente.
>
> A diferença de fundo: os erros padrão do `statsmodels` vêm da **fórmula fechada**, sob a hipótese de erros normais independentes com variância constante. Os nossos vieram do bootstrap, que não assume nada disso. Nestes dados os dois caminhos chegam perto: Grus (2019) registra, ao lado dos valores bootstrap, os erros exatos calculados pela fórmula — 1,19 / 0,080 / 0,127 / 0,998, contra os nossos 1,272 / 0,103 / 0,155 / 1,249.
>
> É tentador ler essa folga como imprecisão do bootstrap e supor que mais reamostras fechariam a diferença. **Não fecham.** Com 400 reamostras em vez de 100, estes dados dão 1,302 / 0,102 / 0,154 / 1,236: os dois do meio mal se mexem e o do termo constante *sobe*, andando para longe de 1,19. Nem poderia ser diferente — o desvio padrão de $B$ réplicas bootstrap não tem viés para cima que se dissolva com $B$ maior. Aumentar $B$ deixa a estimativa mais **estável**, e é só isso: ela converge, mas converge para o número do bootstrap, não para o número da fórmula.
>
> Os dois não convergem um para o outro porque **medem alvos diferentes**. A fórmula responde: *qual seria a dispersão de $\beta$ se os erros fossem normais, independentes e de variância constante?* O bootstrap responde: *qual é a dispersão de $\beta$ quando eu reamostro estes pares?* Só sob as hipóteses da fórmula as duas perguntas têm a mesma resposta. Que elas discordem um pouco é **informação sobre os dados** — o sinal de que alguma hipótese da fórmula não descreve bem este conjunto —, não imprecisão de nenhum dos dois.
>
> O que interessa é o **padrão**, e ele é idêntico pelos dois caminhos: o erro do `doutorado` é uma ordem de grandeza maior que o de `amigos`, e maior que o próprio coeficiente. Quando os dois discordam de verdade, é o bootstrap que merece mais crédito, porque ele nunca dependeu da hipótese que provavelmente é a que caiu.

## Regularização

> **📌 Nota**
>
> Esta seção corresponde a *Regularization*, do capítulo 15 de Grus (2019).

Junte o que as três seções anteriores estabeleceram.

A [seção 12.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/05-qualidade-do-ajuste.html) mostrou que acrescentar variáveis **nunca** piora o R² — nem quando elas são ruído puro. A [seção 12.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/07-erros-padrao-dos-coeficientes.html) mostrou como descobrir que um coeficiente específico é indistinguível de zero. Falta o passo que junta os dois: se o procedimento de ajuste não tem incentivo nenhum para deixar um coeficiente em zero, precisamos **dar** esse incentivo a ele.

É isso que a regularização faz.

> **🔷 Conceito**
>
> Minimizar a soma dos erros ao quadrado é uma instrução com um único objetivo: **errar pouco no conjunto de treino**. Um coeficiente de 0,914 no `doutorado` reduz o erro de treino um fio a mais do que um coeficiente de 0, então o ajuste o adota — mesmo sabendo (como a seção anterior mostrou) que aquele 0,914 é essencialmente ruído.
>
> Regularização muda o objetivo. Em vez de minimizar só o erro, minimizamos **erro mais uma penalidade que cresce com o tamanho dos coeficientes**. Agora um coeficiente só se sustenta se o que ele reduz de erro compensar o que ele custa de penalidade. Um coeficiente que só existe para acomodar ruído não compensa, e encolhe.
>
> O ganho é duplo, e o segundo é fácil de subestimar: quanto menos coeficientes não nulos, mais fácil **entender** o modelo. Se o objetivo é explicar um fenômeno, um modelo esparso com três fatores costuma ser mais útil que um modelo ligeiramente melhor com trezentos.

### A penalidade *ridge*

Na regressão *ridge*, a penalidade é proporcional à soma dos quadrados dos coeficientes — com uma exceção importante, o termo constante:

In [ ]:
import random
random.seed(0)
from scratch.multiple_regression import (
    inputs, error, sqerror_gradient, least_squares_fit_ridge, multiple_r_squared,
)
from scratch.statistics import daily_minutes_good
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
from scratch.linear_algebra import dot, Vector, add

# alpha é um *hiperparâmetro* que controla a força da penalidade.
# Às vezes ele é chamado de "lambda", mas isso já significa algo em Python.
def ridge_penalty(beta: Vector, alpha: float) -> float:
    return alpha * dot(beta[1:], beta[1:])

def squared_error_ridge(x: Vector,
                        y: float,
                        beta: Vector,
                        alpha: float) -> float:
    """erro estimado mais a penalidade ridge sobre beta"""
    return error(x, y, beta) ** 2 + ridge_penalty(beta, alpha)

> **❗ Importante — O `beta[1:]` não é um detalhe**
>
> `beta[0]` fica de fora da penalidade, e o motivo é conceitual, não técnico.
>
> O termo constante não descreve **relação** nenhuma entre variáveis: ele só posiciona o modelo na altura certa. No nosso caso ele vale cerca de 30,5 porque as pessoas passam dezenas de minutos por dia no site — se a variável resposta fosse medida em segundos, ele passaria a valer 1.830, e uma penalidade sobre ele castigaria pesadamente um modelo que não mudou em nada.
>
> Pior: encolher o intercepto em direção a zero significa empurrar todas as previsões em direção a zero, o que é uma afirmação absurda sobre o mundo. Encolher os **outros** coeficientes em direção a zero significa "na dúvida, suponha que esta variável não importa" — uma postura conservadora e defensável. São coisas completamente diferentes, e é por isso que a fatia começa em 1.

O gradiente da penalidade sai direto da derivada de $\alpha \beta_j^2$, com o mesmo zero na primeira posição:

In [ ]:
def ridge_penalty_gradient(beta: Vector, alpha: float) -> Vector:
    """gradiente apenas da penalidade ridge"""
    return [0.] + [2 * alpha * beta_j for beta_j in beta[1:]]

def sqerror_ridge_gradient(x: Vector,
                           y: float,
                           beta: Vector,
                           alpha: float) -> Vector:
    """
    o gradiente correspondente ao i-ésimo termo de erro quadrático,
    incluindo a penalidade ridge
    """
    return add(sqerror_gradient(x, y, beta),
               ridge_penalty_gradient(beta, alpha))

Como a perda é uma soma de duas parcelas, o gradiente é a soma dos dois gradientes — e é literalmente isso que `add` faz.

`error` e `sqerror_gradient`, usados acima, são exatamente as funções da [seção 12.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/03-ajustando-o-modelo.html), importadas de `scratch.multiple_regression`. A partir daí, a mudança conceitual no otimizador é uma só: trocar `sqerror_gradient` por `sqerror_ridge_gradient` dentro do `least_squares_fit` daquela seção. O chute inicial, o laço sobre lotes, o `gradient_step` — tudo o que faz o gradiente descendente ser gradiente descendente fica igual. O pacote `scratch` já traz essa versão pronta, como `least_squares_fit_ridge`.

### O efeito de apertar a penalidade

Com `alpha = 0` não há penalidade nenhuma, e devemos recuperar o ajuste da seção 12.3. Conforme `alpha` cresce, o ajuste piora e os coeficientes encolhem:

In [ ]:
print(f"{'alpha':>6}  {'constante':>10}{'amigos':>9}{'horas':>9}{'doutorado':>11}"
      f"{'|beta|²':>10}{'R²':>8}")

resultados = {}
for alpha in [0.0, 0.1, 1.0, 10.0]:
    random.seed(0)
    b = least_squares_fit_ridge(inputs, daily_minutes_good, alpha,
                                0.001, 5000, 25)
    resultados[alpha] = b
    print(f"{alpha:6.1f}  {b[0]:10.3f}{b[1]:9.3f}{b[2]:9.3f}{b[3]:11.3f}"
          f"{dot(b[1:], b[1:]):10.3f}"
          f"{multiple_r_squared(inputs, daily_minutes_good, b):8.4f}")

Leia a coluna do `doutorado` de cima para baixo: **0,914 → 0,538 → 0,104 → −0,005**.

> **❗ Importante — O arco se fecha aqui**
>
> O coeficiente que evapora primeiro é exatamente o que a [seção 12.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/07-erros-padrao-dos-coeficientes.html) identificou como indistinguível de ruído — o mesmo que a [seção 12.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/02-hipoteses-do-minimos-quadrados.html) já havia flagrado como quase redundante em relação a `num_amigos`.
>
> E o mecanismo é justamente o que se esperava. Um coeficiente com erro padrão grande é um coeficiente que os dados quase não sustentam: mexer nele muda pouquíssimo o erro de treino. Quando a penalidade entra, esse "pouquíssimo" perde para o custo de mantê-lo grande, e ele desce. Já os coeficientes de `amigos` e `horas`, que os dados sustentam com firmeza, resistem: com `alpha = 1` eles ainda estão em 0,897 e −1,677, perto dos valores originais.
>
> Regularização é, nesse sentido, uma **seleção automática de variáveis por sobrevivência**: quem não se sustenta, cai. Ela chega à mesma conclusão que os erros padrão e os valores-p da seção anterior, sem calcular nenhum dos dois — e sem que ninguém precise olhar uma tabela e decidir o que remover.

Repare também no que o R² faz nessa tabela. Ele **desce** conforme `alpha` sobe: 0,680 → 0,680 → 0,675 → 0,560. Isso é a contrapartida obrigatória do que a [seção 12.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/05-qualidade-do-ajuste.html) demonstrou. O modelo sem penalidade é, por definição, o que minimiza o erro de treino; qualquer coisa que o afaste dali piora o erro de treino. A aposta da regularização é que essa piora no treino compra uma melhora nos dados que o modelo ainda não viu — que é a única melhora que importa. Repare que `alpha = 0,1` custou 0,0003 de R² e derrubou o coeficiente do `doutorado` pela metade: barato.

> **⚠️ Atenção — Ponha as variáveis na mesma escala antes de regularizar**
>
> A penalidade *ridge* soma os quadrados dos coeficientes, e o tamanho de um coeficiente depende da **unidade** da variável dele.
>
> Grus (2019) dá o exemplo: se você trocasse "anos de experiência" por "séculos de experiência", os valores da coluna ficariam 100 vezes menores, o coeficiente de mínimos quadrados ficaria 100 vezes maior — e passaria a ser penalizado 10.000 vezes mais, sendo exatamente o mesmo modelo.
>
> A consequência prática é que regularização só faz sentido depois de colocar as variáveis numa escala comum. A ferramenta é o `rescale` da [seção 7.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/06-reescalonamento.html), que subtrai a média e divide pelo desvio padrão de cada coluna. Note que os nossos dados **não** passaram por isso: `amigos` vai de 1 a 49 e `doutorado` é 0 ou 1, escalas bem diferentes. A tabela acima serve para mostrar o mecanismo, e o resultado é interpretável porque nós sabemos o que cada variável significa — mas não é assim que se faz de verdade.

### O outro caminho: *lasso*

A penalidade *ridge* usa a soma dos **quadrados**. Uma alternativa é usar a soma dos **valores absolutos**:

In [ ]:
def lasso_penalty(beta, alpha):
    return alpha * sum(abs(beta_i) for beta_i in beta[1:])

A diferença de comportamento é maior do que a diferença de código sugere. Enquanto a *ridge* encolhe todos os coeficientes proporcionalmente — deixando-os pequenos, mas raramente exatamente zero (repare que nem mesmo com `alpha = 10` o `doutorado` chegou a zero: parou em −0,005) —, a *lasso* tende a **zerar** coeficientes de vez. Isso a torna a ferramenta certa para aprender modelos esparsos, em que boa parte das variáveis some do modelo e as sobreviventes são todas legíveis.

E aqui o livro esbarra num limite honesto: **a *lasso* não é tratável por gradiente descendente**, então não conseguimos resolvê-la do zero. O motivo é visível na definição: $|\beta_j|$ não tem derivada em $\beta_j = 0$ — a função tem um bico ali. E o bico está exatamente no ponto para o qual a *lasso* quer empurrar os coeficientes, ou seja, não é uma patologia que dê para ignorar. Resolvê-la exige métodos de otimização (descida por coordenadas, operadores proximais) que estão fora do escopo daqui.

> **📌 Nota — Dois limites de escopo, de naturezas diferentes**
>
> Vale reparar que este é o segundo limite de escopo deste capítulo, e que os dois não são o mesmo tipo de limite.
>
> A fórmula fechada da [seção 12.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/03-ajustando-o-modelo.html) ficou de fora porque exigiria construir álgebra linear que o livro não construiu — uma decisão de orçamento de páginas. A *lasso* fica de fora porque a ferramenta de otimização que o livro construiu **não serve** para ela: não é questão de páginas, é questão de o gradiente descendente ser a ferramenta errada para uma função com bico.
>
> Reconhecer qual dos dois casos você está enfrentando é uma habilidade prática. No primeiro, insistir eventualmente funciona. No segundo, insistir nunca funciona.

### O que este capítulo amarra

Oito seções, uma linha só de raciocínio. O modelo de várias variáveis é o produto escalar da [seção 12.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/01-o-modelo.html), e ajustá-lo não exigiu nada além do gradiente descendente que o [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html) já tinha: a [seção 12.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/03-ajustando-o-modelo.html) passou de uma variável explicativa para três sem mudar uma linha do otimizador. Se o capítulo terminasse aí, seria um capítulo curto sobre generalizar uma fórmula.

O que o alonga é tudo o que vem **depois** de ter os coeficientes na mão. A [seção 12.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/04-interpretando-o-modelo.html) mostrou que ler um coeficiente é afirmar "mantendo tudo o mais constante" — uma comparação que os dados podem simplesmente não conter, e que nos nossos não contêm para o `doutorado`. A [seção 12.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/05-qualidade-do-ajuste.html) mostrou que a métrica óbvia para arbitrar isso é cega: o R² sobe até com colunas de ruído puro, e uma métrica que nunca piora não serve de critério para deixar nada de fora. A [seção 12.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/06-digressao-o-bootstrap.html) foi buscar por outro caminho — reamostrar os próprios dados — o que a teoria clássica não nos deixava calcular; a [seção 12.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/07-erros-padrao-dos-coeficientes.html) converteu isso num erro padrão por coeficiente e, com ele, disse com todas as letras que o coeficiente do `doutorado` é indistinguível de zero. Esta seção fechou o circuito: em vez de olhar a tabela e remover a variável à mão, mudou o objetivo que o ajuste persegue, para que o coeficiente que os dados não sustentam encolha sozinho.

Três peças ficam para o resto do livro, e nenhuma delas é sobre regressão. **O bootstrap** estima a incerteza de qualquer estatística que você consiga escrever como função de uma lista — e a reamostragem com reposição que ele inventou para isso reaparece com outro objetivo nas florestas aleatórias do [Capítulo 14](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/index.html), lá para treinar modelos em vez de medir incerteza. **O erro padrão** separa efeito de ruído, e é o que permite dizer "esta variável não se sustenta" com um número em vez de com uma impressão. **A regularização** é a resposta padrão ao sobreajuste do [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/03-overfitting-e-underfitting.html), e ela viaja intacta para o próximo modelo: o [Capítulo 13](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/index.html) troca a soma dos erros ao quadrado por outra função de perda, perde a fórmula fechada de vez e passa a depender inteiramente do gradiente descendente — e a penalidade desta seção o acompanha sem nenhuma alteração conceitual, uma parcela somada à perda, um gradiente somado ao gradiente. É por isso que o `scikit-learn` mantém a regularização **ligada por padrão** no `LogisticRegression`, detalhe que costuma pegar de surpresa quem chega lá sem ter passado por aqui.

> **💡 Dica — Na prática: `scikit-learn`**
>
> ```python
> from sklearn.linear_model import Ridge, Lasso, ElasticNet, RidgeCV
> from sklearn.preprocessing import StandardScaler
> from sklearn.pipeline import make_pipeline
>
> modelo = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
> modelo.fit(X, y)
> ```
>
> Quatro coisas que este código traz e o nosso não:
>
> - **`StandardScaler` dentro de um `Pipeline`.** É o `rescale` do Capítulo 7, e o `Pipeline` garante que ele seja ajustado só no treino — sem isso, informação do conjunto de teste vaza para o pré-processamento. Regularização sem reescalonamento é o erro mais comum desta família de modelos, e é por isso que ele aparece primeiro aqui.
> - **`Lasso`**, que o `scikit-learn` resolve por descida por coordenadas — o método que contorna o bico em zero e que nós não temos. É a peça que faltava.
> - **`ElasticNet`**, que combina as duas penalidades e é o que se usa na dúvida.
> - **`RidgeCV`** (e `LassoCV`), que escolhem o `alpha` por validação cruzada em vez de por tentativa e erro. Note que `alpha` é um **hiperparâmetro**: não sai do ajuste, tem que ser escolhido de fora, e a maneira honesta de escolhê-lo é medir desempenho em dados que o modelo não viu — o [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/04-correcao.html) de novo.
>
> Um aviso de nomenclatura, para quando você chegar ao [Capítulo 13](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/index.html): o `alpha` do `Ridge` do `scikit-learn` tem o mesmo papel do nosso, mas o `LogisticRegression` usa `C`, que é o **inverso** da força da penalidade — ali, `C` pequeno significa penalidade forte. Confundir os dois inverte o modelo por completo, e é um erro que não gera exceção nenhuma.

## Leituras adicionais

A seção "For Further Exploration" do capítulo 15 de Grus (2019) faz três sugestões.

A primeira é a mais importante: regressão tem uma teoria rica e extensa por trás, e este é um dos pontos do livro em que vale abrir um livro-texto de estatística de verdade. James et al. (2021) é o ponto de partida usual — o capítulo sobre regressão linear trata de colinearidade, seleção de variáveis e diagnóstico de resíduos com muito mais cuidado do que cabe aqui. Hastie et al. (2009) vai fundo em regularização, incluindo o *lasso* que a [seção 12.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/08-regularizacao.html) só menciona de passagem.

A segunda: o módulo [`linear_model`](https://scikit-learn.org/stable/modules/linear_model.html) do `scikit-learn` traz `LinearRegression`, além de `Ridge`, `Lasso` e `ElasticNet` — as versões industriais de tudo o que este capítulo constrói à mão.

A terceira: o [`statsmodels`](https://www.statsmodels.org/) é outro módulo Python com modelos de regressão, e é o que mais se aproxima do que este capítulo faz na [seção 12.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/07-erros-padrao-dos-coeficientes.html). Enquanto o `scikit-learn` devolve coeficientes e mais nada, o `statsmodels` devolve a tabela inteira que um estatístico espera ver: erro padrão, estatística *t*, valor-p e intervalo de confiança para cada coeficiente.

## Referências

- **Grus**. *Data Science from Scratch: First Principles with Python*. 2nd ed.. O'Reilly Media. 2019.
- **Hastie; Tibshirani; Friedman**. *The Elements of Statistical Learning*. 2nd ed.. Springer. 2009.
- **James; Witten; Hastie; Tibshirani**. *An Introduction to Statistical Learning*. 2nd ed.. Springer. 2021.